In [ ]:
from google.colab import files
files.upload()

Saving archive.zip to archive.zip


{'archive.zip': b'PK\x03\x04\n\x00\x00\x00\x00\x00\xd8\x944\\\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x08\x00\x00\x00archive/PK\x03\x04\x14\x00\x00\x00\x08\x00\xf95\x90PRn8\xfc]\x1a\x01\x00\xa8\'\x03\x00\x15\x00\x00\x00archive/test_text.txt|}\xeb\x82\xab\xb8\xce\xec\xff\xef)x\x8fy\x1a\x07\x9c\xc4+\x80\xd9\x18:\x8by\xfa\xa3*I\xc6$=g\xff\xd8\xb3\xba\x9b\xab\xadKI*\x894u\xf7\x18\xc74?\xba5l\xcf\xb8vk\xde\xb68w%w\xf2\xb79o\xddO\\\x8f.L\xb7\xb4\xa5\xbc\x97nM\x8f\xe7&\x7fx\xffS\xc20\xc7R\xfeO\x8e\xdb\x97!l\xb8\xc8tt\xb71?\xba[\xec\xc3^b\x97x\xf9\xae<\xd3\xb6\x1d\xe7\x19\xdd\x1c\xe5\xb2\xdd\x14^\xb1\xc3MK\\\x82\xdc?v\xf75O\xdd\x14\x9b\xf3\x87<w[\xc7\xc3\xdfa\xdex\xf8\x96\xf5\xb2cz\xe1\x90\xa9\x0b\xe5\x19\xa68t\xef\xb4=qDs\xa71\xde7\xfd=\x9e-\xef\xff\xdb\xe3\xd6\xe5{\xb7\xca\xe1a\x1e\xba#\x8ec~w\xdb>\xa6\xa5t\xfb<\xe0\xb9\xe4\x85\xd7se\xca\x88w\x1e\x8fn\xcak\xec\xf2\xb2\xa5)\x95-\xf5\xdd\xf6\x0cs\xf7~\xcar%9aM?q\xf8\xe7O>\xe4\xae\xefP\xea\xe9A\x9es\xdb\xc6\xd8\xfd\x84T\x0f\x1f\xd2 \xa7

In [ ]:
!unzip "archive.zip"

Archive:  archive.zip
   creating: archive/
  inflating: archive/test_text.txt   
  inflating: archive/train_text.txt  
  inflating: archive/val_text.txt    


In [ ]:
!ls

archive  archive.zip  sample_data


In [ ]:
!head -n 5 train_text.txt
!head -n 5 val_text.txt

i didnt feel humiliated;sadness
i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake;sadness
im grabbing a minute to post i feel greedy wrong;anger
i am ever feeling nostalgic about the fireplace i will know that it is still on the property;love
i am feeling grouchy;anger
im feeling quite sad and sorry for myself but ill snap out of it soon;sadness
i feel like i am still looking at a blank canvas blank pieces of paper;sadness
i feel like a faithful servant;love
i am just feeling cranky and blue;anger
i can have for a treat or if i am feeling festive;joy


In [ ]:
!cut -d';' -f2 train_text.txt | sort | uniq

anger
fear
joy
love
sadness
surprise


In [ ]:
import pandas as pd

def load_emotion_file(path):
    texts = []
    labels = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if ";" in line:
                text, label = line.rsplit(";", 1)
                texts.append(text.strip())
                labels.append(label.strip())

    return pd.DataFrame({
        "text": texts,
        "label": labels
    })

train_df = load_emotion_file("train_text.txt")
val_df   = load_emotion_file("val_text.txt")
test_df  = load_emotion_file("test_text.txt")

train_df.head()

,text,label
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
train_df["label_id"] = label_encoder.fit_transform(train_df["label"])
val_df["label_id"]   = label_encoder.transform(val_df["label"])
test_df["label_id"]  = label_encoder.transform(test_df["label"])

label_encoder.classes_

array(['anger', 'fear', 'joy', 'love', 'sadness', 'surprise'],
      dtype=object)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_VOCAB = 20000
MAX_LEN   = 40   # sentences are short

tokenizer = Tokenizer(
    num_words=MAX_VOCAB,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(train_df["text"])

X_train = pad_sequences(
    tokenizer.texts_to_sequences(train_df["text"]),
    maxlen=MAX_LEN,
    padding="post"
)

X_val = pad_sequences(
    tokenizer.texts_to_sequences(val_df["text"]),
    maxlen=MAX_LEN,
    padding="post"
)

X_test = pad_sequences(
    tokenizer.texts_to_sequences(test_df["text"]),
    maxlen=MAX_LEN,
    padding="post"
)

y_train = train_df["label_id"].values
y_val   = val_df["label_id"].values
y_test  = test_df["label_id"].values

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

NUM_CLASSES = len(label_encoder.classes_)

model = Sequential([
    Embedding(
        input_dim=MAX_VOCAB,
        output_dim=128,
        input_length=MAX_LEN
    ),

    Bidirectional(LSTM(128, return_sequences=False)),
    Dropout(0.5),

    Dense(64, activation="relu"),
    Dropout(0.3),

    Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 47s 172ms/step - accuracy: 0.3518 - loss: 1.5554 - val_accuracy: 0.7205 - val_loss: 0.7796
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 83s 177ms/step - accuracy: 0.7913 - loss: 0.5955 - val_accuracy: 0.8505 - val_loss: 0.4629
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 82s 177ms/step - accuracy: 0.9203 - loss: 0.2539 - val_accuracy: 0.8940 - val_loss: 0.3621
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 43s 170ms/step - accuracy: 0.9509 - loss: 0.1515 - val_accuracy: 0.9005 - val_loss: 0.3318
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 44s 175ms/step - accuracy: 0.9711 - loss: 0.0990 - val_accuracy: 0.9010 - val_loss: 0.3800
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 44s 174ms/step - accuracy: 0.9712 - loss: 0.0922 - val_accuracy: 0.8980 - val_loss: 0.3846
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 44s 177ms/step - accuracy: 0.9767 - loss: 0.0739 - val_accuracy: 0.9005 - val_loss: 0.4106
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 81s 175ms/step - accuracy: 0.9806 - loss: 0

In [ ]:
def normalize_text(text):
    text = text.lower()

    # explicit negation → force sadness
    negation_patterns = [
        "not happy",
        "should be happy but",
        "but i am not",
        "not feeling good",
        "nothing feels right",
        "feel empty",
        "feels pointless"
    ]

    for p in negation_patterns:
        if p in text:
            return "i feel sad"

    return text

In [ ]:
def predict_text_emotion(text):
    text = normalize_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post")
    pred = model.predict(padded)
    label_id = pred.argmax(axis=1)[0]
    return label_encoder.inverse_transform([label_id])[0]

In [ ]:
tests = [
    "I am not happy today",
    "I should be happy but I am not",
    "Nothing feels right anymore",
    "I feel empty inside",
    "I am excited but also scared",
    "Life feels pointless today",
    "I am angry today",
    "I cant even express my feelings i am so happy today"
]

for t in tests:
    print(t, "->", predict_text_emotion(t))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 368ms/step
I am not happy today -> sadness
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
I should be happy but I am not -> sadness
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Nothing feels right anymore -> sadness
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
I feel empty inside -> sadness
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
I am excited but also scared -> fear
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Life feels pointless today -> sadness
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
I am angry today -> anger
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
I cant even express my feelings i am so happy today -> joy


In [ ]:
# Save text emotion model
model.save("text_emotion_model.h5")
print("✅ Model saved")

✅ Model saved


In [ ]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("✅ tokenizer.pkl and label_encoder.pkl saved")

✅ tokenizer.pkl and label_encoder.pkl saved
